# **Reinforcement Learning project: Part 2**

## **Task 5 - Train and test your policies**

Train two agents with your algorithm of choice (the best performing), on the source and target domains respectively. Then, test each model and report its average return over 50 test episodes. In particular, report results for the following “training→test” configurations:
* source→source, (this will be just for reference, since the goal is to obtain optimal performance in the target environment)
* source→target (lower bound),
* target→target (upper bound).
Test with different hyperparameters and report the best results found together with the parameters used.  The results will be the upper bound and lower bound for the following Domain Randomization phase.

**Guiding Questions**
* Why do we expect lower performances from the “source→target” configuration w.r.t. the “target→target”?
* If higher performances can be reached by training on the target environment directly, what prevents us from doing so (in a sim-to-real setting)?

### **project setup**
Let's start by setting up the environment for the project, we will be cloning the github repository and importing the necessary libraries for the tasks we will handle

In [ ]:
# cloning the given github reporistory
!git clone https://github.com/lambdavi/FAIML-RL-26.git

%cd FAIML-RL-26/part2/panda-gym
!pip install -e .

%cd ..


fatal: destination path 'FAIML-RL-26' already exists and is not an empty directory.
/content/FAIML-RL-26/part2/panda-gym
Obtaining file:///content/FAIML-RL-26/part2/panda-gym
  Preparing metadata (setup.py) ... done
  Attempting uninstall: panda_gym
    Found existing installation: panda_gym 3.0.8
    Uninstalling panda_gym-3.0.8:
      Successfully uninstalled panda_gym-3.0.8
  Running setup.py develop for panda_gym
/content/FAIML-RL-26/part2


In [ ]:
!pip install stable-baselines3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.5/187.5 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 952.1/952.1 kB 8.0 MB/s eta 0:00:00
  Attempting uninstall: gymnasium
    Found existing installation: gymnasium 1.3.0
    Uninstalling gymnasium-1.3.0:
      Successfully uninstalled gymnasium-1.3.0


**Imports**

In [ ]:
import gymnasium as gym
import panda_gym
import numpy as np
from stable_baselines3 import SAC

from stable_baselines3.common.callbacks import CheckpointCallback
from stable_baselines3.common.monitor import Monitor

from google.colab import drive

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


**Create the environments**

In [ ]:
np.random.seed(42)
# SOURCE DOMAIN
source_env = gym.make(
    "PandaPush-v3",
    render_mode="rgb_array",
    reward_type="dense",
    type="source"
)
source_env = Monitor(source_env)
# TARGET DOMAIN
target_env = gym.make(
    "PandaPush-v3",
    render_mode="rgb_array",
    reward_type="dense",
    type="target"
)
# change the reward type ?
# target_env.unwrapped.action_noise = 0.1
target_env = Monitor(target_env)

Created object with mass: 1.0
Created object with mass: 5.0


**Evaluation function for 50 episodes**

In [ ]:
def evaluate(model, env, n_episodes=50):

    returns = []

    for episode in range(n_episodes):

        obs, _ = env.reset()
        done = False
        ep_reward = 0

        while not done:
            action, _ = model.predict(obs, deterministic=True)
            obs, reward, terminated, truncated, _ = env.step(action)

            done = terminated or truncated
            ep_reward += reward

        returns.append(ep_reward)

    mean_return = np.mean(returns)
    std_return = np.std(returns)

    return mean_return, std_return

###**Hyperparameter configurations :**
lr = 1e-4 | gamma = 0.95

In [ ]:
np.random.seed(42)

learning_rates = [1e-4]
gammas = [0.95]

results = []

for lr in learning_rates:
    for gamma in gammas:

        print(f"\nTesting lr={lr} | gamma={gamma}")

        # target
        model_target = SAC(
            "MultiInputPolicy",
            target_env,
            learning_rate=lr,
            gamma=gamma,
            batch_size=256,
            verbose=0,
            seed = 42
        )

        model_target.learn(total_timesteps=500000)

        # evaluation
        tt_mean, tt_std = evaluate(model_target, target_env)
        print("tt_mean : ")
        print(tt_mean)
        print("tt_std : ")
        print(tt_std)

        # source
        model_source = SAC(
            "MultiInputPolicy",
            source_env,
            learning_rate=lr,
            gamma=gamma,
            batch_size=256,
            verbose=0,
            seed = 42
        )

        model_source.learn(total_timesteps=500000)

        # evaluations
        ss_mean, ss_std = evaluate(model_source, source_env)
        st_mean, st_std = evaluate(model_source, target_env)
        print("ss_mean : ")
        print(ss_mean)
        print("ss_std : ")
        print(ss_std)
        print("\n st_mean : ")
        print(st_mean)
        print("\n st_std : ")
        print(st_std)



        # results
        results.append({
            "learning_rate": lr,
            "gamma": gamma,

            "source->source mean": ss_mean,
            "source->source std": ss_std,

            "source->target mean": st_mean,
            "source->target std": st_std,

            "target->target mean": tt_mean,
            "target->target std": tt_std,
        })



Testing lr=0.0001 | gamma=0.95
tt_mean : 
-3.1358675236720592
tt_std : 
3.280593211815436


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


ss_mean : 
-3.015298761893064
ss_std : 
2.0494121549722997

 st_mean : 
-3.7151250865496697

 st_std : 
1.7470446212630133


lr = 1e-4 | gamma = 0.98

In [ ]:
np.random.seed(42)

learning_rates = [1e-4]
gammas = [0.98]

results = []

for lr in learning_rates:
    for gamma in gammas:

        print(f"\nTesting lr={lr} | gamma={gamma}")

        # source
        model_source = SAC(
            "MultiInputPolicy",
            source_env,
            learning_rate=lr,
            gamma=gamma,
            batch_size=256,
            verbose=0,
            seed = 42
        )

        model_source.learn(total_timesteps=500000)

        # evaluations
        ss_mean, ss_std = evaluate(model_source, source_env)
        st_mean, st_std = evaluate(model_source, target_env)
        print("ss_mean : ")
        print(ss_mean)
        print("ss_std : ")
        print(ss_std)
        print("\n st_mean : ")
        print(st_mean)
        print("\n st_std : ")
        print(st_std)

        # target
        model_target = SAC(
            "MultiInputPolicy",
            target_env,
            learning_rate=lr,
            gamma=gamma,
            batch_size=256,
            verbose=0,
            seed = 42
        )

        model_target.learn(total_timesteps=500000)

        # evaluation
        tt_mean, tt_std = evaluate(model_target, target_env)
        print("tt_mean : ")
        print(tt_mean)
        print("tt_std : ")
        print(tt_std)

        # results
        results.append({
            "learning_rate": lr,
            "gamma": gamma,

            "source->source mean": ss_mean,
            "source->source std": ss_std,

            "source->target mean": st_mean,
            "source->target std": st_std,

            "target->target mean": tt_mean,
            "target->target std": tt_std,
        })



Testing lr=0.0001 | gamma=0.98
ss_mean : 
-2.944279440399259
ss_std : 
2.0872833894521023

 st_mean : 
-3.0434706661850215

 st_std : 
2.2017991097714043


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


tt_mean : 
-3.183426103517413
tt_std : 
2.0629212957943626


lr = 1e-4 | gamma = 0.99

In [ ]:
np.random.seed(42)

learning_rates = [1e-4]
gammas = [0.99]

results = []

for lr in learning_rates:
    for gamma in gammas:

        print(f"\nTesting lr={lr} | gamma={gamma}")

        # source
        model_source = SAC(
            "MultiInputPolicy",
            source_env,
            learning_rate=lr,
            gamma=gamma,
            batch_size=256,
            verbose=0,
            seed = 42
        )

        model_source.learn(total_timesteps=500000)

        # evaluations
        ss_mean, ss_std = evaluate(model_source, source_env)
        st_mean, st_std = evaluate(model_source, target_env)
        print("ss_mean : ")
        print(ss_mean)
        print("ss_std : ")
        print(ss_std)
        print("\n st_mean : ")
        print(st_mean)
        print("\n st_std : ")
        print(st_std)

        # target
        model_target = SAC(
            "MultiInputPolicy",
            target_env,
            learning_rate=lr,
            gamma=gamma,
            batch_size=256,
            verbose=0,
            seed = 42
        )

        model_target.learn(total_timesteps=500000)

        # evaluation
        tt_mean, tt_std = evaluate(model_target, target_env)
        print("tt_mean : ")
        print(tt_mean)
        print("tt_std : ")
        print(tt_std)

        # results
        results.append({
            "learning_rate": lr,
            "gamma": gamma,

            "source->source mean": ss_mean,
            "source->source std": ss_std,

            "source->target mean": st_mean,
            "source->target std": st_std,

            "target->target mean": tt_mean,
            "target->target std": tt_std,
        })



Testing lr=0.0001 | gamma=0.99
ss_mean : 
-2.767603596504778
ss_std : 
2.1908932193346526

 st_mean : 
-3.510433337315917

 st_std : 
1.8570172133199616


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


tt_mean : 
-3.256854021437466
tt_std : 
2.005166772298219


lr = 3e-4 | gamma = 0.95

In [ ]:
np.random.seed(42)

learning_rates = [3e-4]
gammas = [0.95]

results = []

for lr in learning_rates:
    for gamma in gammas:

        print(f"\nTesting lr={lr} | gamma={gamma}")

        # target
        model_target = SAC(
            "MultiInputPolicy",
            target_env,
            learning_rate=lr,
            gamma=gamma,
            batch_size=256,
            verbose=0,
            seed = 42
        )

        model_target.learn(total_timesteps=500000)

        # evaluation
        tt_mean, tt_std = evaluate(model_target, target_env)
        print("tt_mean : ")
        print(tt_mean)
        print("tt_std : ")
        print(tt_std)

        # source
        model_source = SAC(
            "MultiInputPolicy",
            source_env,
            learning_rate=lr,
            gamma=gamma,
            batch_size=256,
            verbose=0,
            seed = 42
        )

        model_source.learn(total_timesteps=500000)

        # evaluations
        ss_mean, ss_std = evaluate(model_source, source_env)
        st_mean, st_std = evaluate(model_source, target_env)
        print("ss_mean : ")
        print(ss_mean)
        print("ss_std : ")
        print(ss_std)
        print("\n st_mean : ")
        print(st_mean)
        print("\n st_std : ")
        print(st_std)



        # results
        results.append({
            "learning_rate": lr,
            "gamma": gamma,

            "source->source mean": ss_mean,
            "source->source std": ss_std,

            "source->target mean": st_mean,
            "source->target std": st_std,

            "target->target mean": tt_mean,
            "target->target std": tt_std,
        })



Testing lr=0.0003 | gamma=0.95
tt_mean : 
-0.3432301988545805
tt_std : 
0.25310362395992014


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


ss_mean : 
-0.3390728198364377
ss_std : 
0.21414283323021338

 st_mean : 
-0.7780115218274295

 st_std : 
0.9244793685472843


lr = 3e-4 | gamma = 0.98

In [ ]:
np.random.seed(42)

learning_rates = [3e-4]
gammas = [0.98]

results = []

for lr in learning_rates:
    for gamma in gammas:

        print(f"\nTesting lr={lr} | gamma={gamma}")

        # target
        model_target = SAC(
            "MultiInputPolicy",
            target_env,
            learning_rate=lr,
            gamma=gamma,
            batch_size=256,
            verbose=0,
            seed = 42
        )

        model_target.learn(total_timesteps=500000)

        # evaluation
        tt_mean, tt_std = evaluate(model_target, target_env)
        print("tt_mean : ")
        print(tt_mean)
        print("tt_std : ")
        print(tt_std)

        # source
        model_source = SAC(
            "MultiInputPolicy",
            source_env,
            learning_rate=lr,
            gamma=gamma,
            batch_size=256,
            verbose=0,
            seed = 42
        )

        model_source.learn(total_timesteps=500000)

        # evaluations
        ss_mean, ss_std = evaluate(model_source, source_env)
        st_mean, st_std = evaluate(model_source, target_env)
        print("ss_mean : ")
        print(ss_mean)
        print("ss_std : ")
        print(ss_std)
        print("\n st_mean : ")
        print(st_mean)
        print("\n st_std : ")
        print(st_std)



        # results
        results.append({
            "learning_rate": lr,
            "gamma": gamma,

            "source->source mean": ss_mean,
            "source->source std": ss_std,

            "source->target mean": st_mean,
            "source->target std": st_std,

            "target->target mean": tt_mean,
            "target->target std": tt_std,
        })



Testing lr=0.0003 | gamma=0.98
tt_mean : 
-0.3676735005620867
tt_std : 
0.30004738213087256


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


ss_mean : 
-1.4109565461799503
ss_std : 
1.7269859395141849

 st_mean : 
-1.6246233795210718

 st_std : 
1.7443464261085384


lr = 3e-4 | gamma = 0.99

In [ ]:
np.random.seed(42)

learning_rates = [3e-4]
gammas = [0.99]

results = []

for lr in learning_rates:
    for gamma in gammas:

        print(f"\nTesting lr={lr} | gamma={gamma}")

        # target
        model_target = SAC(
            "MultiInputPolicy",
            target_env,
            learning_rate=lr,
            gamma=gamma,
            batch_size=256,
            verbose=0,
            seed = 42
        )

        model_target.learn(total_timesteps=500000)

        # evaluation
        tt_mean, tt_std = evaluate(model_target, target_env)
        print("tt_mean : ")
        print(tt_mean)
        print("tt_std : ")
        print(tt_std)

        # source
        model_source = SAC(
            "MultiInputPolicy",
            source_env,
            learning_rate=lr,
            gamma=gamma,
            batch_size=256,
            verbose=0,
            seed = 42
        )

        model_source.learn(total_timesteps=500000)

        # evaluations
        ss_mean, ss_std = evaluate(model_source, source_env)
        st_mean, st_std = evaluate(model_source, target_env)
        print("ss_mean : ")
        print(ss_mean)
        print("ss_std : ")
        print(ss_std)
        print("\n st_mean : ")
        print(st_mean)
        print("\n st_std : ")
        print(st_std)



        # results
        results.append({
            "learning_rate": lr,
            "gamma": gamma,

            "source->source mean": ss_mean,
            "source->source std": ss_std,

            "source->target mean": st_mean,
            "source->target std": st_std,

            "target->target mean": tt_mean,
            "target->target std": tt_std,
        })



Testing lr=0.0003 | gamma=0.99
tt_mean : 
-0.4359949986822903
tt_std : 
0.3295546342728004


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


ss_mean : 
-2.1234219350107013
ss_std : 
1.9005566793320945

 st_mean : 
-2.61018764635548

 st_std : 
2.095752069370492
